# 🚀 Meter Reading Pipeline - M1 → M2 + Smart Rotate → M3 → M4

## Pipeline Overview:

```
Input Image
    ↓
M1: Image Preprocessing (Denoise, Enhance, Adjust)
    ↓
M2: Object Detection + Smart Rotation (Auto-rotate for better OCR)
    ↓
M3: OCR Recognition (CRNN + biLSTM + CTC)
    ↓
M4: Post-processing (Validate, Format, Score)
    ↓
Output: Meter Reading with Confidence
```

---

## 1️⃣ Install Dependencies & Mount Drive

In [ ]:
# Install dependencies
!pip install torch torchvision opencv-python matplotlib tqdm -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Mount Google Drive
from google.colab import drive
import os

print("\n🔗 Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Drive mounted!")

## 2️⃣ Load Trained OCR Model

Update the path to your trained model below.

In [ ]:
# Path to your trained model (update this!)
MODEL_PATH = "/content/drive/MyDrive/ocr_training_checkpoints/best_model.pth"

# Check if model exists
if os.path.exists(MODEL_PATH):
    print(f"✅ Model found: {MODEL_PATH}")
else:
    print(f"❌ Model not found: {MODEL_PATH}")
    print("\nPlease update MODEL_PATH to point to your trained model.")
    print("It should be in: /content/drive/MyDrive/ocr_training_checkpoints/best_model.pth")

## 3️⃣ Define Pipeline Modules

In [ ]:
import cv2
import numpy as np
from typing import Dict, List, Tuple, Optional
import json
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### M1: Image Preprocessing

In [ ]:
class M1_ImagePreprocessor:
    """M1: Image Preprocessing Module"""
    
    def __init__(self, target_size=(512, 512)):
        self.target_size = target_size
    
    def preprocess(self, image):
        # Denoise
        image = self._denoise(image)
        
        # Enhance contrast
        image = self._enhance_contrast(image)
        
        # Adjust brightness
        image = self._adjust_brightness(image)
        
        # Resize
        image = cv2.resize(image, self.target_size)
        
        return image
    
    def _denoise(self, image):
        if len(image.shape) == 3:
            return cv2.bilateralFilter(image, 9, 75, 75)
        else:
            return cv2.fastNlMeansDenoising(image, None, 10, 7, 21)
    
    def _enhance_contrast(self, image):
        if len(image.shape) == 3:
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            l = clahe.apply(l)
            enhanced = cv2.merge([l, a, b])
            return cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)
        else:
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            return clahe.apply(image)
    
    def _adjust_brightness(self, image):
        if len(image.shape) == 3:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
            h, s, v = cv2.split(hsv)
            v = cv2.add(v, 50)
            v = np.clip(v, 0, 255).astype('uint8')
            adjusted = cv2.merge([h, s, v])
            return cv2.cvtColor(adjusted, cv2.COLOR_HSV2RGB)
        else:
            adjusted = cv2.add(image, 50)
            return np.clip(adjusted, 0, 255).astype('uint8')

print("✅ M1: Image Preprocessing Module loaded")

### M2: Object Detection + Smart Rotation

In [ ]:
class M2_SmartRotator:
    """M2: Object Detection + Smart Rotation Module"""
    
    def __init__(self, debug=False):
        self.debug = debug
    
    def process(self, image):
        # Detect meter region
        meter_region = self._detect_meter_region(image)
        
        # Calculate rotation angle
        angle = self._calculate_rotation_angle(meter_region)
        
        # Rotate image
        rotated = self._rotate_image(meter_region, angle)
        
        return rotated, angle
    
    def _detect_meter_region(self, image):
        # Convert to grayscale
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()
        
        # Edge detection
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blurred, 50, 150)
        
        # Find contours
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return image
        
        # Get largest contour
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        
        # Extract region with padding
        padding = 10
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(image.shape[1], x + w + padding)
        y2 = min(image.shape[0], y + h + padding)
        
        return image[y1:y2, x1:x2]
    
    def _calculate_rotation_angle(self, image):
        # Convert to grayscale
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()
        
        # Threshold
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        # Detect text lines
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (20, 1))
        dilated = cv2.dilate(binary, kernel, iterations=2)
        
        # Find contours
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours:
            return 0.0
        
        # PCA for angle calculation
        all_points = np.vstack([cnt for cnt in contours])
        mean, eigenvectors = cv2.PCACompute(all_points.astype(np.float32), mean=None)
        
        # Calculate angle
        angle = np.degrees(np.arctan2(eigenvectors[0, 1], eigenvectors[0, 0]))
        
        # Adjust to [-45, 45] range
        if angle > 45:
            angle -= 90
        elif angle < -45:
            angle += 90
        
        return angle
    
    def _rotate_image(self, image, angle):
        height, width = image.shape[:2]
        center = (width // 2, height // 2)
        
        # Get rotation matrix
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        
        # Calculate new dimensions
        cos = np.abs(rotation_matrix[0, 0])
        sin = np.abs(rotation_matrix[0, 1])
        new_width = int((height * sin) + (width * cos))
        new_height = int((height * cos) + (width * sin))
        
        # Adjust matrix
        rotation_matrix[0, 2] += (new_width / 2) - center[0]
        rotation_matrix[1, 2] += (new_height / 2) - center[1]
        
        # Rotate
        border_color = (255, 255, 255) if len(image.shape) == 3 else 255
        rotated = cv2.warpAffine(image, rotation_matrix, (new_width, new_height),
                                flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT,
                                borderValue=border_color)
        return rotated

print("✅ M2: Smart Rotation Module loaded")

### M3: OCR Recognition

In [ ]:
class M3_OCRRecognizer:
    """M3: OCR Recognition Module"""
    
    def __init__(self, model_path, device):
        self.model_path = model_path
        self.device = device
        
        # Character mappings
        self.char_map = "0123456789"
        self.label_to_char = {i: c for i, c in enumerate(self.char_map)}
        self.blank_idx = 10
        
        # Load model
        self.model = self._load_model()
        self.model.eval()
        
        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((64, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
    
    def _load_model(self):
        # Define CRNN architecture
        class CRNN(nn.Module):
            def __init__(self):
                super(CRNN, self).__init__()
                self.cnn = nn.Sequential(
                    nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
                    nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
                    nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2),
                    nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2, 1), (2, 1)),
                    nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
                )
                self.rnn = nn.LSTM(1024, 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)
                self.fc = nn.Linear(512, 11)
            
            def forward(self, x):
                conv = self.cnn(x)
                b, c, h, w = conv.size()
                conv = conv.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
                out, _ = self.rnn(conv)
                return self.fc(out).permute(1, 0, 2)
        
        model = CRNN().to(self.device)
        
        if os.path.exists(self.model_path):
            checkpoint = torch.load(self.model_path, map_location=self.device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✅ Model loaded from: {self.model_path}")
        else:
            print(f"⚠️  Model not found: {self.model_path}")
        
        return model
    
    def recognize(self, image):
        # Convert to PIL
        if len(image.shape) == 3:
            pil_image = Image.fromarray(image).convert('L')
        else:
            pil_image = Image.fromarray(image)
        
        # Transform
        image_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)
        
        # Recognize
        with torch.no_grad():
            predictions = self.model(image_tensor)
        
        # Decode
        text, confidence = self._decode(predictions)
        
        return {'text': text, 'confidence': confidence}
    
    def _decode(self, predictions):
        probs = predictions.softmax(dim=-1)
        pred_indices = predictions.argmax(dim=-1).permute(1, 0).cpu().numpy()[0]
        
        chars = []
        confidences = []
        prev = None
        
        for t, idx in enumerate(pred_indices):
            if idx != self.blank_idx and idx != prev:
                chars.append(self.label_to_char[idx])
                confidences.append(probs[t, 0, idx].item())
            prev = idx
        
        text = ''.join(chars)
        confidence = np.mean(confidences) if confidences else 0.0
        
        return text, confidence

# Initialize OCR module
if os.path.exists(MODEL_PATH):
    m3 = M3_OCRRecognizer(MODEL_PATH, device)
    print("✅ M3: OCR Recognition Module loaded")
else:
    print("❌ Please update MODEL_PATH first!")

### M4: Post-processing

In [ ]:
class M4_PostProcessor:
    """M4: Post-processing Module"""
    
    def __init__(self, expected_length=4):
        self.expected_length = expected_length
    
    def process(self, ocr_result, metadata=None):
        text = ocr_result['text']
        raw_confidence = ocr_result['confidence']
        
        # Validate
        validation = self._validate(text)
        
        # Correct format
        corrected_text = self._correct(text)
        
        # Calculate final confidence
        final_confidence = self._calculate_confidence(
            raw_confidence, validation, corrected_text != text
        )
        
        return {
            'text': corrected_text,
            'raw_text': text,
            'confidence': final_confidence,
            'raw_confidence': raw_confidence,
            'is_valid': validation['valid'],
            'errors': validation['errors'],
            'metadata': metadata or {}
        }
    
    def _validate(self, text):
        errors = []
        
        if len(text) != self.expected_length:
            errors.append(f"Expected {self.expected_length} digits, got {len(text)}")
        
        if not text.isdigit():
            errors.append("Contains non-digit characters")
        
        return {'valid': len(errors) == 0, 'errors': errors}
    
    def _correct(self, text):
        corrected = text
        
        # Common OCR corrections
        corrections = {'O': '0', 'I': '1', 'l': '1', 'S': '5', 'Z': '2', 'B': '8'}
        for wrong, correct in corrections.items():
            corrected = corrected.replace(wrong, correct)
        
        # Keep only digits
        corrected = ''.join(c for c in corrected if c.isdigit())
        
        # Pad/truncate
        if len(corrected) < self.expected_length:
            corrected = corrected.zfill(self.expected_length)
        elif len(corrected) > self.expected_length:
            corrected = corrected[:self.expected_length]
        
        return corrected
    
    def _calculate_confidence(self, raw_conf, validation, was_corrected):
        confidence = raw_conf
        
        if not validation['valid']:
            confidence *= 0.8
        if was_corrected:
            confidence *= 0.9
        
        return max(0.0, min(1.0, confidence))

m4 = M4_PostProcessor()
print("✅ M4: Post-processing Module loaded")

## 4️⃣ Complete Pipeline Function

In [ ]:
def process_meter_reading(image_path, visualize=True, save_dir=None):
    """
    Complete pipeline: M1 -> M2 + Smart Rotate -> M3 -> M4
    """
    print("\n" + "="*60)
    print("🚀 METER READING PIPELINE")
    print("="*60)
    print(f"Input: {image_path}")
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    
    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    results = {}
    
    # ===== M1: Preprocessing =====
    print("\n[M1] Image Preprocessing...")
    m1 = M1_ImagePreprocessor()
    preprocessed = m1.preprocess(image)
    results['m1'] = preprocessed
    
    # ===== M2: Smart Rotation =====
    print("[M2] Smart Rotation...")
    m2 = M2_SmartRotator()
    rotated, angle = m2.process(preprocessed)
    results['m2'] = {'image': rotated, 'angle': angle}
    print(f"    Rotation angle: {angle:.2f}°")
    
    # ===== M3: OCR =====
    print("[M3] OCR Recognition...")
    ocr_result = m3.recognize(rotated)
    print(f"    Raw text: {ocr_result['text']}")
    print(f"    Confidence: {ocr_result['confidence']:.4f}")
    results['m3'] = ocr_result
    
    # ===== M4: Post-processing =====
    print("[M4] Post-processing...")
    final = m4.process(ocr_result, {'rotation_angle': angle})
    print(f"    Final text: {final['text']}")
    print(f"    Final confidence: {final['confidence']:.4f}")
    print(f"    Valid: {final['is_valid']}")
    results['final'] = final
    
    # Visualize
    if visualize:
        visualize_pipeline(image, preprocessed, rotated, final, save_dir)
    
    print("\n" + "="*60)
    print("✅ PIPELINE COMPLETE!")
    print("="*60)
    
    return results

def visualize_pipeline(original, preprocessed, rotated, result, save_dir=None):
    """Visualize all pipeline stages"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Row 1: Images
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Original Image')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(preprocessed)
    axes[0, 1].set_title('M1: Preprocessed')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(rotated)
    axes[0, 2].set_title(f'M2: Rotated ({result["metadata"]["rotation_angle"]:.2f}°)')
    axes[0, 2].axis('off')
    
    # Row 2: Results
    axes[1, 0].axis('off')
    axes[1, 0].text(0.5, 0.7, f'M3 Raw: {result["raw_text"]}',
                      ha='center', va='center', fontsize=14, weight='bold')
    axes[1, 0].text(0.5, 0.3, f'Conf: {result["raw_confidence"]:.4f}',
                      ha='center', va='center', fontsize=12)
    
    axes[1, 1].axis('off')
    axes[1, 1].text(0.5, 0.7, f'M4 Final: {result["text"]}',
                      ha='center', va='center', fontsize=16, weight='bold', color='green')
    axes[1, 1].text(0.5, 0.3, f'Conf: {result["confidence"]:.4f}',
                      ha='center', va='center', fontsize=12)
    
    axes[1, 2].axis('off')
    status = '✅ VALID' if result['is_valid'] else '❌ INVALID'
    axes[1, 2].text(0.5, 0.6, status,
                      ha='center', va='center', fontsize=16, weight='bold')
    if result['errors']:
        axes[1, 2].text(0.5, 0.3, '\n'.join(result['errors'][:3]),
                          ha='center', va='center', fontsize=9, color='red')
    
    plt.tight_layout()
    
    if save_dir:
        save_path = os.path.join(save_dir, 'pipeline_result.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n💾 Visualization saved to: {save_path}")
    else:
        plt.show()
    
    plt.close()

print("✅ Pipeline functions loaded!")

## 5️⃣ Test Pipeline

Upload a test image or use the path to your dataset images.

In [ ]:
# Test on a single image
# UPDATE THIS PATH to your test image
TEST_IMAGE = "/content/m4_ocr_dataset_black_digits/images/crop_meter4_00000_0001e09f7ad5442a832f7b5efb74bf2c.jpg"

# Check if image exists
if os.path.exists(TEST_IMAGE):
    print(f"✅ Test image found: {TEST_IMAGE}")
else:
    print(f"❌ Test image not found: {TEST_IMAGE}")
    print("\nPlease upload a test image or update TEST_IMAGE path.")

In [ ]:
# Run pipeline
result = process_meter_reading(
    image_path=TEST_IMAGE,
    visualize=True,
    save_dir='/content/pipeline_output'
)

# Print final result
print("\n" + "="*60)
print("FINAL READING")
print("="*60)
print(f"Meter Reading: {result['final']['text']}")
print(f"Confidence: {result['final']['confidence']:.4f}")
print(f"Rotation Angle: {result['m2']['angle']:.2f}°")
print(f"Valid: {result['final']['is_valid']}")

## 6️⃣ Batch Processing

Process multiple images at once.

In [ ]:
def batch_process(image_paths, output_dir):
    """Process multiple images"""
    results = []
    
    for i, img_path in enumerate(image_paths, 1):
        print(f"\n{'='*60}")
        print(f"Processing {i}/{len(image_paths)}: {img_path}")
        print(f"{'='*60}")
        
        try:
            result = process_meter_reading(
                img_path,
                visualize=False,
                save_dir=os.path.join(output_dir, f'result_{i}')
            )
            results.append({
                'image': img_path,
                'reading': result['final']['text'],
                'confidence': result['final']['confidence'],
                'valid': result['final']['is_valid']
            })
        except Exception as e:
            print(f"❌ Error: {e}")
    
    return results

# Example: Process first 10 images from dataset
# image_dir = "/content/m4_ocr_dataset_black_digits/images"
# image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir)[:10]]
# results = batch_process(image_paths, '/content/batch_results')

print("✅ Batch processing function loaded!")

## ✅ Summary

### Pipeline Stages:

1. **M1: Image Preprocessing**
   - Denoise (bilateral filter)
   - Enhance contrast (CLAHE)
   - Adjust brightness
   - Resize to target size

2. **M2: Object Detection + Smart Rotation**
   - Detect meter region (edge detection + contours)
   - Calculate rotation angle (PCA on text lines)
   - Smart rotate to upright position

3. **M3: OCR Recognition**
   - Load trained CRNN + biLSTM model
   - Recognize text from rotated image
   - Return raw text with confidence

4. **M4: Post-processing**
   - Validate reading (length, digits)
   - Correct common OCR errors (O→0, I→1, etc.)
   - Calculate final confidence score

### Usage:
```python
result = process_meter_reading(
    image_path='path/to/image.jpg',
    visualize=True,
    save_dir='output'
)

print(f"Reading: {result['final']['text']}")
print(f"Confidence: {result['final']['confidence']}")
```

---

**🎉 Pipeline ready to use!**